# Численные методы — вариант 20

Ноутбук собран из GitHub-проекта `Elizaveta120000/-cislen.metod`.

Исходные файлы проекта:

- `variant20.py` — основной вариант 20: отделение корней, половинное деление, простая итерация, комбинированный метод хорд и касательных, проверка через SciPy.
- `korni_programs.py` — дополнительные программы по теме уточнения корней.

Источник: https://github.com/Elizaveta120000/-cislen.metod  
Коммит: `84635f5`, 2026-06-08.


## Подготовка

Если в вашей Jupyter-среде не установлены библиотеки для графиков или SciPy, можно раскомментировать и выполнить следующую строку.


In [ ]:
# %pip install matplotlib scipy


In [ ]:
import math

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib не установлен: графики будут пропущены.")

try:
    from scipy.optimize import brentq as scipy_brentq
except ImportError:
    scipy_brentq = None
    print("scipy не установлен: в задании 5 будет использован встроенный метод бисекции.")


## Часть 1. Основной вариант 20

Решается уравнение:

\[
\ln(x + 6.1) = 2\sin(x - 1.4)
\]

Для численных методов используем функцию:

\[
f(x)=\ln(x + 6.1)-2\sin(x - 1.4)
\]


In [ ]:
# Сама функция и её производные.
def f(x):
    return math.log(x + 6.1) - 2 * math.sin(x - 1.4)


def df(x):
    return 1 / (x + 6.1) - 2 * math.cos(x - 1.4)


def d2f(x):
    return -1 / (x + 6.1) ** 2 + 2 * math.sin(x - 1.4)


# Отрезок, на котором отделён корень.
A = -2.5
B = -2.0


### Задание 1. Отделение корней


In [ ]:
def zadanie1():
    print("=" * 55)
    print("ЗАДАНИЕ 1. Отделение корней")
    print("=" * 55)

    print(" x      f(x)        знак")
    x = -6.0
    prev = None
    while x <= 10.0001:
        y = f(x)
        znak = "+" if y > 0 else "-" if y < 0 else "0"
        smena = ""
        if prev is not None and prev[1] * y < 0:
            smena = "  <-- здесь меняется знак, есть корень"
        print("%6.2f  %+10.5f   %s%s" % (x, y, znak, smena))
        prev = (x, y)
        x += 0.5

    print("\nБерём для дальнейшей работы корень на отрезке [%.1f; %.1f]" % (A, B))
    print(
        "Проверка: f(%.1f)=%+.5f, f(%.1f)=%+.5f -> разные знаки\n"
        % (A, f(A), B, f(B))
    )

    if plt is None:
        print("График не построен: установите matplotlib и перезапустите ячейку.")
        return

    xs = [(-6.0 + i * 0.02) for i in range(int((10 - (-6)) / 0.02))]
    y1 = [math.log(xx + 6.1) for xx in xs]
    y2 = [2 * math.sin(xx - 1.4) for xx in xs]
    yf = [f(xx) for xx in xs]

    plt.figure(figsize=(9, 5))
    plt.plot(xs, y1, label="y = ln(x+6.1)")
    plt.plot(xs, y2, label="y = 2 sin(x-1.4)")
    plt.plot(xs, yf, "--", label="f(x)=ln(x+6.1)-2 sin(x-1.4)")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.grid(True)
    plt.legend()
    plt.title("Вариант 20: отделение корней")
    plt.show()


In [ ]:
zadanie1()


### Задание 2. Метод половинного деления

Точность: \(10^{-3}\).


In [ ]:
def zadanie2(eps=1e-3):
    print("=" * 55)
    print("ЗАДАНИЕ 2. Метод половинного деления (eps=%g)" % eps)
    print("=" * 55)
    a, b = A, B
    n = 0
    print(" n     a         b         c        f(c)      (b-a)/2")
    while (b - a) / 2 > eps:
        c = (a + b) / 2
        print(
            "%2d  %8.5f  %8.5f  %8.5f  %+9.5f  %.6f"
            % (n, a, b, c, f(c), (b - a) / 2)
        )
        if f(a) * f(c) < 0:
            b = c
        else:
            a = c
        n += 1
    koren = (a + b) / 2
    print("Корень x = %.4f (за %d делений)\n" % (koren, n))
    return koren


In [ ]:
k2 = zadanie2()


### Задание 3. Метод простой итерации

Точность: \(10^{-6}\). Уравнение приводится к виду:

\[
x = \varphi(x), \quad \varphi(x)=x-\frac{f(x)}{M}
\]


In [ ]:
def zadanie3(eps=1e-6):
    print("=" * 55)
    print("ЗАДАНИЕ 3. Метод простой итерации (eps=%g)" % eps)
    print("=" * 55)

    M = max(abs(df(A)), abs(df(B)), abs(df((A + B) / 2)))
    M = M * 1.05
    print("Берём M = %.4f, тогда phi(x) = x - f(x)/M" % M)

    def phi(x):
        return x - f(x) / M

    x = (A + B) / 2
    n = 0
    print(" n      x_n            phi(x_n)        |x_n+1 - x_n|")
    while True:
        xn = phi(x)
        print("%2d   %.9f   %.9f   %.2e" % (n, x, xn, abs(xn - x)))
        if abs(xn - x) < eps:
            break
        x = xn
        n += 1
    print("Корень x = %.6f (за %d итераций)\n" % (xn, n))
    return xn


In [ ]:
k3 = zadanie3()


### Задание 4. Комбинированный метод хорд и касательных

Точность: \(10^{-6}\).


In [ ]:
def zadanie4(eps=1e-6):
    print("=" * 55)
    print("ЗАДАНИЕ 4. Комбинированный метод хорд и касательных (eps=%g)" % eps)
    print("=" * 55)
    a, b = A, B
    print(
        "f''(%.1f)=%+.4f, f''(%.1f)=%+.4f  -> вторая производная знак не меняет"
        % (a, d2f(a), b, d2f(b))
    )

    n = 0
    print(" n   хорда(a)       касательная(b)   длина [a;b]")
    while abs(b - a) > eps:
        if f(a) * d2f(a) > 0:
            a = a - f(a) / df(a)
            b = b - f(b) * (a - b) / (f(a) - f(b))
        else:
            b = b - f(b) / df(b)
            a = a - f(a) * (b - a) / (f(b) - f(a))
        print("%2d   %.9f   %.9f   %.2e" % (n, a, b, abs(b - a)))
        n += 1
    koren = (a + b) / 2
    print("Корень x = %.6f (за %d шагов)\n" % (koren, n))
    return koren


In [ ]:
k4 = zadanie4()


### Задание 5. Проверка через инструментальный пакет

Если установлен SciPy, используется `scipy.optimize.brentq`. Если SciPy недоступен, ячейка выполнит встроенный вариант бисекции, чтобы ноутбук всё равно можно было прогнать.


In [ ]:
def local_bisect(func, a, b, eps):
    fa = func(a)
    fb = func(b)
    if fa * fb > 0:
        raise ValueError("На концах отрезка функция должна иметь разные знаки.")

    n = 0
    while (b - a) / 2 > eps:
        c = (a + b) / 2
        fc = func(c)
        if fa * fc <= 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc
        n += 1
    return (a + b) / 2, n


def zadanie5(eps=1e-6):
    print("=" * 55)
    print("ЗАДАНИЕ 5. Решение через пакет SciPy (brentq)")
    print("=" * 55)

    if scipy_brentq is not None:
        koren = scipy_brentq(f, A, B, xtol=eps)
        print("Метод: scipy.optimize.brentq")
    else:
        koren, n = local_bisect(f, A, B, eps)
        print("SciPy не найден, использован встроенный метод бисекции (%d шагов)." % n)

    print("Корень x = %.8f" % koren)
    print("Проверка: f(x) = %.2e\n" % f(koren))
    return koren


In [ ]:
k5 = zadanie5()


### Сопоставление результатов


In [ ]:
print("=" * 55)
print("СОПОСТАВЛЕНИЕ РЕЗУЛЬТАТОВ")
print("=" * 55)
print("Половинное деление (1e-3):      x = %.4f" % k2)
print("Простая итерация   (1e-6):      x = %.6f" % k3)
print("Хорды + касательные(1e-6):      x = %.6f" % k4)
print("Пакет/проверка     (1e-6):      x = %.6f" % k5)


## Часть 2. Дополнительные программы из `korni_programs.py`

Ниже перенесены дополнительные функции и примеры из второго файла проекта.


In [ ]:
lg = math.log10
ln = math.log
sin = math.sin
cos = math.cos


def bisect_general(func, a, b, eps):
    n = 0
    while (b - a) / 2 > eps:
        c = (a + b) / 2
        if func(a) * func(c) < 0:
            b = c
        else:
            a = c
        n += 1
    return (a + b) / 2, n


def iterate(phi, x0, eps, maxn=1000):
    x = x0
    for n in range(maxn):
        xn = phi(x)
        if abs(xn - x) < eps:
            return xn, n + 1
        x = xn
    return x, maxn


### Скрин 1. Отделение корней


In [ ]:
def skrin1():
    print("=== СКРИН 1: отделение корней ===")

    fa = lambda x: lg(x) + 6 - x ** 2
    fb = lambda x: x * sin(x) - 1

    print("а) lg x + 6 - x^2:")
    for x in [1e-7, 1e-6, 1e-5, 1, 2, 2.5, 2.55, 3]:
        print("   f(%g) = %+.5f" % (x, fa(x)))
    print("   -> корни около 1e-6 и на [2.5; 2.55]")

    print("б) x sin x - 1:")
    for x in [0, 1, 1.1, 1.2, 1.5708]:
        print("   f(%g) = %+.5f" % (x, fb(x)))
    print("   -> наименьший ненулевой корень на [1.1; 1.2]\n")

    if plt is None:
        print("График не построен: установите matplotlib и перезапустите ячейку.")
        return

    xs = [0.01 + i * 0.01 for i in range(400)]
    plt.figure(figsize=(9, 4))
    plt.subplot(1, 2, 1)
    plt.plot(xs, [lg(x) + 6 for x in xs], label="lg x + 6")
    plt.plot(xs, [x ** 2 for x in xs], label="x^2")
    plt.title("a) lg x + 6 = x^2")
    plt.legend()
    plt.grid(True)
    plt.ylim(0, 8)

    xs2 = [i * 0.02 for i in range(1, 200)]
    plt.subplot(1, 2, 2)
    plt.plot(xs2, [sin(x) for x in xs2], label="sin x")
    plt.plot(xs2, [1 / x for x in xs2], label="1/x")
    plt.title("б) x sin x = 1")
    plt.legend()
    plt.grid(True)
    plt.ylim(-1, 3)
    plt.tight_layout()
    plt.show()


In [ ]:
skrin1()


### Скрин 2. Половинное деление для \(x\sin x - 1 = 0\)


In [ ]:
def skrin2():
    print("=== СКРИН 2: половинное деление x sin x - 1 = 0 (eps=1e-4) ===")
    func = lambda x: x * sin(x) - 1
    x, n = bisect_general(func, 1.1, 1.2, 1e-4)
    print("   корень x = %.4f  (за %d делений)\n" % (x, n))


In [ ]:
skrin2()


### Скрин 5.2. Простая итерация для \(x\sin x - 1 = 0\)


In [ ]:
def skrin3():
    print("=== СКРИН 5.2: простая итерация x sin x - 1 = 0 (eps=1e-5) ===")
    phi = lambda x: 1 / sin(x)
    x = 1.1
    n = 0
    print("    n      x_n          x_(n+1)        |разность|")
    while True:
        xn = phi(x)
        print("   %2d  %.7f   %.7f   %.2e" % (n, x, xn, abs(xn - x)))
        if abs(xn - x) < 1e-5:
            break
        x = xn
        n += 1
        if n > 1000:
            raise RuntimeError("Метод не сошёлся за 1000 итераций.")
    print("   корень x = %.5f  (за %d итераций)\n" % (xn, n + 1))


In [ ]:
skrin3()
